# Time series cross validation: label timing and a level shift

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/time-series/time_series_cross_validation.ipynb)

All data are generated locally. This notebook runs on CPU without credentials or repository files.

Dependencies: numpy, matplotlib, pillow, scikit-learn. If needed, install these packages with `%pip install` before running the imports.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Keep generated figures in a temporary folder for this notebook session.
figure_folder = TemporaryDirectory()
IMG = Path(figure_folder.name)
COLOURS = ['#167d8d', '#e6a23b', '#526b98']
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 120})

def show_table(columns, rows):
    header = '| ' + ' | '.join(columns) + ' |'
    separator = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = ['| ' + ' | '.join(map(str, row)) + ' |' for row in rows]
    display(Markdown('\n'.join([header, separator] + body)))

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from matplotlib.animation import PillowWriter

## Forecast rows and labels

An origin t uses the current observation and 23 earlier observations. Its label is the average of observations t+1 through t+12. The generator's `shift` parameter changes only the level shift at index 900. Seeded noise is otherwise identical for the shifted and unshifted control.

The development set includes only labels ending before origin 1400. Future evaluation begins at 1400 and uses a model fitted once on development data. Current and past observations become available as inputs at each future origin.

In [ ]:
# Generate the seasonal series and construct retrospective supervised rows.
def forecast_data(seed=0, shift=3.0):
    rng = np.random.default_rng(seed)
    n = 1800
    z = np.zeros(n)
    for t in range(1, n):
        z[t] = 0.82 * z[t - 1] + rng.normal(0, 0.5)
    t = np.arange(n)
    series = 2 * np.sin(2 * np.pi * t / 48) + z + np.where(t >= 900, shift, 0.0)
    origins = np.arange(24, n - 12)
    X = np.array([series[o - np.arange(24)] for o in origins])
    y = np.array([series[o + 1:o + 13].mean() for o in origins])
    return (series, origins, X, y)

In [ ]:
series, origins, X, y = forecast_data(seed=0, shift=3)
dev = np.flatnonzero(origins + 12 < 1400)
future = np.flatnonzero(origins >= 1400)
assert origins[dev[-1]] + 12 < origins[future[0]]
print('Development origins:', origins[dev[0]], 'to', origins[dev[-1]])
print('Future origins:', origins[future[0]], 'to', origins[future[-1]])

## Model selection within each split

For every ridge penalty, the function averages five fold MSEs. It selects the smallest mean MSE and then takes its square root. This is not the mean of five fold RMSEs. Each `StandardScaler` is fitted inside the training pipeline, so validation values cannot affect its means or scales.

Selection uses development folds only. The selected penalty is refitted on all permitted development rows before the future holdout is scored.

In [ ]:
# Select a penalty within each of the three split procedures.
def forecast_experiment():
    records = []
    example = None
    for seed in range(5):
        series, origins, X, y = forecast_data(seed)
        dev = np.flatnonzero(origins + 12 < 1400)
        future = np.flatnonzero(origins >= 1400)
        methods = {'Shuffled': KFold(5, shuffle=True, random_state=42), 'Expanding': TimeSeriesSplit(5), 'Gap 12': TimeSeriesSplit(5, gap=12)}
        for name, splitter in methods.items():
            scores = []
            for alpha in [0.01, 1.0, 100.0, 10000.0]:
                errors = []
                for tr, va in splitter.split(dev):
                    assert not np.intersect1d(tr, va).size
                    if name == 'Gap 12':
                        assert origins[dev[tr[-1]]] + 12 < origins[dev[va[0]]]
                    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
                    model.fit(X[dev[tr]], y[dev[tr]])
                    errors.append(mean_squared_error(y[dev[va]], model.predict(X[dev[va]])))
                scores.append(np.mean(errors))
            selected = int(np.argmin(scores))
            alpha = [0.01, 1.0, 100.0, 10000.0][selected]
            model = make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(X[dev], y[dev])
            pred = model.predict(X[future])
            rmse = mean_squared_error(y[future], pred) ** 0.5
            records.append({'seed': seed, 'method': name, 'alpha': alpha, 'cv_rmse': float(scores[selected] ** 0.5), 'future_rmse': float(rmse)})
            if seed == 0 and name == 'Gap 12':
                example = (series, origins, X, y, dev, future, pred)
    return (records, example)

In [ ]:
records, example = forecast_experiment()
selection_table = []
for method in ['Shuffled', 'Expanding', 'Gap 12']:
    rows = [r for r in records if r['method'] == method]
    selection_table.append([method,
        f"{np.mean([r['cv_rmse'] for r in rows]):.3f}",
        f"{np.mean([r['future_rmse'] for r in rows]):.3f}",
        ', '.join(str(r['alpha']) for r in rows)])
show_table(['Split', 'Mean selected CV RMSE', 'Mean future RMSE',
            'Selected penalties by seed'], selection_table)

## Shift and flexible-model controls

The ridge-only result cannot isolate overlap effects. Fixing ridge alpha at 0.01 and removing the shift reveals which fold drives the gap difference. A five-nearest-neighbour model on the same scaled lag features tests a more flexible predictor. Its parameters are fixed; the future holdout remains unused for selection.

In [ ]:
# Compare fixed ridge and KNN models with and without the level shift.
def forecast_controls():
    """Hold model parameters fixed to isolate shift and split effects."""
    records = []
    for shift in [0.0, 3.0]:
        for seed in range(5):
            series, origins, X, y = forecast_data(seed, shift=shift)
            dev = np.flatnonzero(origins + 12 < 1400)
            future = np.flatnonzero(origins >= 1400)
            splits = {'Shuffled': KFold(5, shuffle=True, random_state=42), 'Expanding': TimeSeriesSplit(5), 'Gap 12': TimeSeriesSplit(5, gap=12)}
            for model_name, learner in [('Ridge', Ridge(alpha=0.01)), ('KNN 5', KNeighborsRegressor(5))]:
                for method, splitter in splits.items():
                    fold_errors = []
                    for train, valid in splitter.split(dev):
                        model = make_pipeline(StandardScaler(), learner)
                        model.fit(X[dev[train]], y[dev[train]])
                        fold_errors.append(float(mean_squared_error(y[dev[valid]], model.predict(X[dev[valid]]))))
                    model = make_pipeline(StandardScaler(), learner).fit(X[dev], y[dev])
                    records.append(dict(shift=shift, seed=seed, model=model_name, method=method, fold_mse=fold_errors, cv_rmse=float(np.mean(fold_errors) ** 0.5), future_rmse=float(mean_squared_error(y[future], model.predict(X[future])) ** 0.5)))
    return records

In [ ]:
controls = forecast_controls()
fold_table = []
for shift in [3., 0.]:
    for method in ['Expanding', 'Gap 12']:
        rows = [r for r in controls if r['model'] == 'Ridge'
                and r['shift'] == shift and r['method'] == method]
        fold_means = np.mean([r['fold_mse'] for r in rows], axis=0)
        fold_table.append([f'Shift {shift:g}, {method}']
                          + [f'{value:.3f}' for value in fold_means])
show_table(['Fixed ridge alpha=.01', 'Fold 1', 'Fold 2', 'Fold 3',
            'Fold 4', 'Fold 5'], fold_table)
knn_table = []
for shift in [0., 3.]:
    rows = [r for r in controls if r['model'] == 'KNN 5'
            and r['shift'] == shift and r['method'] == 'Shuffled']
    knn_table.append([f'Shift {shift:g}',
        f"{np.mean([r['cv_rmse'] for r in rows]):.3f}",
        f"{np.mean([r['future_rmse'] for r in rows]):.3f}"])
show_table(['KNN 5', 'Shuffled CV RMSE', 'Future RMSE'], knn_table)

In [ ]:
# Draw the article series and selection-error figures.
def forecast_figures(records, example):
    slug = 'time-series-cross-validation'
    series, origins, X, y, dev, future, pred = example
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.plot(series, color='#526b98', lw=0.8, label='Observed series')
    ax.axvline(900, color='#e6a23b', ls='--', label='Level shift')
    ax.axvline(1400, color='#167d8d', ls='--', label='Future holdout')
    ax.set(xlabel='Time index', ylabel='Value')
    ax.legend(ncol=3, fontsize=9)
    fig.tight_layout()
    fig.savefig(IMG / f'{slug}-series.webp')
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 4))
    names = ['Shuffled', 'Expanding', 'Gap 12']
    for i, name in enumerate(names):
        rows = [r for r in records if r['method'] == name]
        a = np.array([r['cv_rmse'] for r in rows])
        b = np.array([r['future_rmse'] for r in rows])
        ax.bar(i - 0.17, a.mean(), 0.32, color=COLOURS[0], label='Selected CV RMSE' if i == 0 else None)
        ax.bar(i + 0.17, b.mean(), 0.32, color=COLOURS[1], label='Future RMSE' if i == 0 else None)
        ax.scatter(np.full(5, i - 0.17), a, color='black', s=12, zorder=3)
        ax.scatter(np.full(5, i + 0.17), b, color='black', s=12, zorder=3)
    ax.set(xticks=range(3), xticklabels=names, ylabel='RMSE')
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMG / f'{slug}-errors.webp')
    plt.close(fig)

In [ ]:
# Draw per-fold ridge errors and the flexible-model control.
def forecast_control_figure(records):
    """Expose the boundary fold and the flexible-model shuffled-CV control."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for shift, style in [(0.0, '--'), (3.0, '-')]:
        for method, colour in [('Expanding', COLOURS[0]), ('Gap 12', COLOURS[1])]:
            rows = [r for r in records if r['model'] == 'Ridge' and r['shift'] == shift and (r['method'] == method)]
            error = np.mean([r['fold_mse'] for r in rows], axis=0)
            axes[0].plot(range(1, 6), error, marker='o', ls=style, color=colour, label=f'Shift {shift:g}, {method}')
    axes[0].set(xlabel='Chronological fold', ylabel='Mean fold MSE', xticks=range(1, 6))
    axes[0].legend(fontsize=9)
    for i, shift in enumerate([0.0, 3.0]):
        rows = [r for r in records if r['model'] == 'KNN 5' and r['shift'] == shift and (r['method'] == 'Shuffled')]
        axes[1].bar(i - 0.17, np.mean([r['cv_rmse'] for r in rows]), 0.32, label='Shuffled CV' if i == 0 else None, color=COLOURS[0])
        axes[1].bar(i + 0.17, np.mean([r['future_rmse'] for r in rows]), 0.32, label='Future holdout' if i == 0 else None, color=COLOURS[1])
    axes[1].set(xticks=[0, 1], xticklabels=['No level shift', 'Level shift 3'], ylabel='KNN 5 RMSE')
    axes[1].legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(IMG / 'time-series-cross-validation-controls.webp')
    plt.close(fig)

In [ ]:
forecast_figures(records, example)
forecast_control_figure(controls)
for name in ['series', 'errors', 'controls']:
    display(Image(filename=str(IMG / f'time-series-cross-validation-{name}.webp')))

## Expanding folds and available targets

The overview shows the actual five expanding folds and marks index 900. The enlarged boundary separates excluded origins from raw observations that finish an earlier label. Each final training target ends strictly before the first validation origin. The target labels and the grey origin interval refer to different objects.

In [ ]:
# Animate actual fold indices and assert label availability.
def forecast_animation(origins, horizon=12):
    """Animate actual expanding folds and the availability of their last label."""
    from matplotlib.animation import FuncAnimation
    origins = np.asarray(origins)
    splits = list(TimeSeriesSplit(n_splits=5, gap=horizon).split(origins))
    checks = []
    for train, valid in splits:
        last_origin = int(origins[train[-1]])
        first_valid = int(origins[valid[0]])
        label_end = last_origin + horizon
        assert label_end < first_valid
        checks.append({'last_training_origin': last_origin, 'last_training_label_end': label_end, 'first_validation_origin': first_valid, 'excluded_origins': first_valid - last_origin - 1})
    fig, (overview, detail) = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [1.2, 1]})
    teal, amber, blue = ('#167d8d', '#d88a19', '#526b98')
    phases = ['Training origins', 'Last training target', 'First validation forecast']

    def update(frame):
        fold, phase = divmod(frame, 3)
        overview.clear()
        detail.clear()
        for i, (train, valid) in enumerate(splits):
            opacity = 1 if i == fold else 0.18
            overview.plot(origins[train], np.full(len(train), i), color=teal, linewidth=6, alpha=opacity, solid_capstyle='butt')
            overview.plot(origins[valid], np.full(len(valid), i), color=amber, linewidth=6, alpha=opacity, solid_capstyle='butt')
            gap_start = origins[train[-1]] + 0.5
            gap_end = origins[valid[0]] - 0.5
            overview.plot([gap_start, gap_end], [i, i], color='#777777', lw=6, alpha=opacity, solid_capstyle='butt')
        overview.axvline(900, color='#526b98', ls=':', lw=1)
        overview.text(910, 4.25, 'Level shift 900', ha='left', fontsize=9)
        overview.set(xlim=(origins[0] - 10, origins[-1] + 10), ylim=(-0.5, 4.5), yticks=range(5), yticklabels=range(1, 6), xlabel='Forecast origin', ylabel='Expanding fold')
        overview.set_title(f'Fold {fold + 1} of 5: {phases[phase]}')
        row = checks[fold]
        last, end, valid = (row['last_training_origin'], row['last_training_label_end'], row['first_validation_origin'])
        detail.axvspan(last + 0.5, valid - 0.5, color='#dddddd', alpha=0.55)
        detail.scatter([last], [1], color=teal, s=85, alpha=1 if phase == 0 else 0.55, zorder=3)
        detail.plot([last + 1, end], [1, 1], color=teal, linewidth=7, alpha=1 if phase == 1 else 0.4, solid_capstyle='butt')
        detail.scatter([valid], [0], color=amber, s=85, alpha=1 if phase == 2 else 0.4, zorder=3)
        detail.plot([valid + 1, valid + horizon], [0, 0], color=amber, linewidth=7, alpha=1 if phase == 2 else 0.25, solid_capstyle='butt')
        detail.text(valid + 1, 0.3, 'validation target', fontsize=9, color=amber)
        detail.axvline(valid, color=blue, linestyle='--', linewidth=1)
        detail.text(last, 1.3, f'origin {last}', ha='center', fontsize=9)
        detail.text(end, 0.7, f'training target {last + 1} to {end}', ha='right', fontsize=9)
        detail.text(valid, -0.3, f'validation begins {valid}', ha='left', fontsize=9)
        detail.text(last + 1, 1.65, f'{horizon} excluded origins', fontsize=9)
        detail.set(xlim=(last - 3, valid + horizon + 3), ylim=(-0.65, 1.9), yticks=[0, 1], yticklabels=['Validation', 'Last training row'], xlabel='Time index near the boundary')
        detail.set_title(f'Target available before validation: {end} < {valid}', fontsize=11)
        fig.tight_layout()
        return []
    animation = FuncAnimation(fig, update, frames=15, interval=1000 / 3, repeat=True, blit=False)
    return (animation, checks, fig)

In [ ]:
animation, fold_checks, figure = forecast_animation(origins[dev], horizon=12)
animation.save(IMG / 'forecast_folds.gif', writer=PillowWriter(fps=3))
plt.close(figure)
display(Image(filename=str(IMG / 'forecast_folds.gif')))
show_table(['Last training origin', 'Last label endpoint', 'First validation origin'],
    [[r['last_training_origin'], r['last_training_label_end'],
      r['first_validation_origin']] for r in fold_checks])

## Interpretation and exercises

The large gap effect occurs in fold four near the level shift. Without the shift, the gap changes ridge CV RMSE by about 0.001. KNN shuffled CV is optimistic on both series. These controls separate the label-availability rule from model flexibility and the periods used for evaluation.

1. Replace expanding windows with a fixed rolling window, choosing its length within development data.
2. Change the target horizon and update both the row construction and availability checks.
3. Inspect the final training labels in each fold before fitting any model.

References: [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), [Bergmeir, Hyndman and Koo](https://robjhyndman.com/papers/cv-wp.pdf).